# LAB6 – Decision Trees, SVM, and Ensemble Methods (Solutions)

Inspired by the structure of the Lab 5 notebook, this lab focuses on advanced model selection and ensemble methods for the Breast Cancer dataset.

## Table of Contents
- [Chapter I — Decision Tree Hyperparameters](#chapter1)
  - [Exercise 1 — Grid Search with Decision Trees](#ex1)
- [Chapter II — SVM versus Decision Tree](#chapter2)
  - [Exercise 2 — Hyperparameter Search and Comparison](#ex2)
- [Chapter III — Ensemble Learning](#chapter3)
  - [Exercise 3 — Bagging and Voting](#ex3)


---
<a id="chapter1"></a>
# Chapter I — Decision Tree Hyperparameters

In this chapter we revisit decision trees on the Breast Cancer dataset, paying special attention to methodological traps that can hurt generalisation.

<a id="ex1"></a>
## Exercise 1 — Grid Search with Decision Trees

**Objective.** Tune a `DecisionTreeClassifier` with `GridSearchCV` on the Breast Cancer dataset while identifying and correcting at least seven methodological traps.

**Summary of detected traps and corrections**
1. **Data leakage by scaling/splitting in the wrong order** → we split the raw data before any transformation.
2. **Ignoring class imbalance** → we keep stratification in train/test split *and* cross-validation.
3. **Too small cross-validation folds (`cv=2`)** → replaced with a more stable `StratifiedKFold` with 5 folds.
4. **Non-reproducible results** → all stochastic estimators use an explicit `random_state`.
5. **Grid search on the full dataset** → the grid search is restricted to the training set only.
6. **Relying solely on accuracy** → we track accuracy and F1-score to detect imbalance issues.
7. **Unbounded tree growth** → we compare constrained vs unconstrained settings to discuss overfitting.
8. **Forgetting feature names in the tree plot** → pass them explicitly for interpretability.

We follow the numbered steps from the statement and insert short discussions after each critical block.

In [ ]:
# Common imports for the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import perf_counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier, VotingClassifier

plt.style.use('seaborn-v0_8')


### Step 1 — Load and explore the dataset
We begin by loading the Breast Cancer dataset and inspecting its dimensions and class balance.

In [ ]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
class_names = cancer.target_names

n_samples, n_features = X.shape
classes, counts = np.unique(y, return_counts=True)

print(f"Number of observations: {n_samples}")
print(f"Number of features: {n_features}")
print(f"Classes: {dict(zip(class_names, counts))}")

pd.DataFrame(X, columns=feature_names).head()


### Step 2 — Train/test split (with stratification)
A stratified split preserves the malignant/benign ratio in both subsets and avoids the imbalance trap.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

X_train.shape, X_test.shape


### Step 3 — Baseline model
We instantiate a baseline decision tree for reference.

In [ ]:
baseline_tree = DecisionTreeClassifier(random_state=42)
baseline_tree.fit(X_train, y_train)

print(f"Baseline accuracy (train): {baseline_tree.score(X_train, y_train):.3f}")
print(f"Baseline accuracy (test): {baseline_tree.score(X_test, y_test):.3f}")


### Step 4 — Hyperparameter grid
We explore impurity criteria, tree depth, and leaf constraints. Constraining both depth and leaf size mitigates high variance.

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [2, 4, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
param_grid


### Step 5 — Grid search with cross-validation
Using `StratifiedKFold` with five folds stabilises the estimate (instead of the trap `cv=2`).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=True
)

grid_tree.fit(X_train, y_train)


### Step 6 — Best hyperparameters and CV score

In [ ]:
print(f"Best parameters: {grid_tree.best_params_}")
print(f"Best CV accuracy: {grid_tree.best_score_:.4f}")


### Step 7 — Evaluation on the test set
Besides accuracy we also report the F1-score to verify that the class balance trap is mitigated.

In [ ]:
best_tree = grid_tree.best_estimator_

y_pred_tree = best_tree.predict(X_test)

test_metrics_tree = {
    'accuracy': accuracy_score(y_test, y_pred_tree),
    'precision': precision_score(y_test, y_pred_tree),
    'recall': recall_score(y_test, y_pred_tree),
    'f1': f1_score(y_test, y_pred_tree)
}

print(pd.Series(test_metrics_tree).round(4))
print('\nClassification report:\n')
print(classification_report(y_test, y_pred_tree, target_names=class_names))


### Step 8 — Visualise the optimal tree

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    best_tree,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    impurity=True,
    rounded=True,
    fontsize=9
)
plt.title('Optimal Decision Tree (Grid Search)')
plt.show()


### Discussion — Bias/variance and pathological hyperparameters
- **Deeper trees with tiny leaves** (`max_depth=None`, `min_samples_leaf=1`) almost memorise the training data and yield near-perfect training accuracy but slightly lower cross-validation/test performance → classic high-variance regime.
- **Excessively small `min_samples_split`** (e.g. 1) is rejected by scikit-learn because at least two samples are required to split a node. Conceptually, forcing splits with very few samples inflates variance and makes predictions unstable.
- The tuned model balances both aspects: a moderate depth and minimum leaf size provide the best trade-off between bias (underfitting) and variance (overfitting).


In [ ]:
overfit_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

overfit_tree.fit(X_train, y_train)

train_score = overfit_tree.score(X_train, y_train)
cv_score = cross_val_score(
    overfit_tree,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
).mean()

test_score = overfit_tree.score(X_test, y_test)

print(f"Training accuracy (overfitted tree): {train_score:.4f}")
print(f"Cross-validated accuracy (same params): {cv_score:.4f}")
print(f"Test accuracy (overfitted tree): {test_score:.4f}")


---
<a id="chapter2"></a>
# Chapter II — SVM versus Decision Tree

We now compare the tuned decision tree with an optimised Support Vector Machine.

<a id="ex2"></a>
## Exercise 2 — Hyperparameter Search and Comparison
We build pipelines for both models, perform grid searches, and compare them on an independent test set in terms of F1-score, inference time, and interpretability.

In [ ]:
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42))
])

dt_pipeline = Pipeline([
    ('identity', 'passthrough'),
    ('tree', DecisionTreeClassifier(random_state=42))
])

svm_param_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf'],
    'svc__gamma': ['scale', 'auto']
}

dt_param_grid = {
    'tree__criterion': ['gini', 'entropy', 'log_loss'],
    'tree__max_depth': [2, 4, 6, 8, 10, None],
    'tree__min_samples_split': [2, 5, 10],
    'tree__min_samples_leaf': [1, 2, 4]
}


In [ ]:
svm_grid = GridSearchCV(
    svm_pipeline,
    param_grid=svm_param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

svm_grid.fit(X_train, y_train)

best_svm = svm_grid.best_estimator_
print(f"Best SVM params: {svm_grid.best_params_}")
print(f"Best CV F1 (SVM): {svm_grid.best_score_:.4f}")

dt_grid = GridSearchCV(
    dt_pipeline,
    param_grid=dt_param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

dt_grid.fit(X_train, y_train)

best_dt_pipeline = dt_grid.best_estimator_
print(f"Best Tree params: {dt_grid.best_params_}")
print(f"Best CV F1 (Tree): {dt_grid.best_score_:.4f}")


In [ ]:
def evaluate_model(name, model, X_test, y_test):
    start = perf_counter()
    y_pred = model.predict(X_test)
    elapsed_ms = (perf_counter() - start) * 1000
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'inference_time_ms': elapsed_ms,
        'y_pred': y_pred
    }
    return metrics

metrics_dt = evaluate_model('Decision Tree (best)', best_dt_pipeline, X_test, y_test)
metrics_svm = evaluate_model('SVM (best)', best_svm, X_test, y_test)

comparison_df = pd.DataFrame([metrics_dt, metrics_svm]).drop(columns='y_pred').set_index('model')
comparison_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, metrics) in zip(axes, [('Decision Tree', metrics_dt), ('SVM', metrics_svm)]):
    ConfusionMatrixDisplay.from_predictions(y_test, metrics['y_pred'], display_labels=class_names, ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} — Confusion Matrix')
plt.tight_layout()
plt.show()


### Interpretation
- The SVM benefits from feature scaling and typically yields a slightly higher F1-score at the cost of longer inference due to support vectors.
- The decision tree remains faster and more interpretable thanks to its explicit rules (the tree visualised earlier).
- Depending on operational constraints (latency vs transparency), either model might be preferable.

---
<a id="chapter3"></a>
# Chapter III — Ensemble Learning

We extend the comparison to bagging ensembles and a hybrid voting classifier.

<a id="ex3"></a>
## Exercise 3 — Bagging and Voting
Steps: create bagging ensembles for SVM and Decision Tree, optimise their hyperparameters, and combine them via soft voting.

In [ ]:
bagging_svm = BaggingClassifier(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(probability=True, random_state=42))
    ]),
    random_state=42,
    n_jobs=-1
)

bagging_tree = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42,
    n_jobs=-1
)

bagging_svm_grid = {
    'estimator__svc__C': [0.1, 1, 10],
    'estimator__svc__kernel': ['linear', 'rbf'],
    'estimator__svc__gamma': ['scale', 'auto'],
    'n_estimators': [10, 25],
    'max_samples': [0.8, 1.0]
}

bagging_tree_grid = {
    'estimator__criterion': ['gini', 'entropy'],
    'estimator__max_depth': [3, 5, 10, None],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'n_estimators': [25, 50, 100],
    'max_samples': [0.6, 0.8, 1.0]
}


In [ ]:
bag_svm_search = GridSearchCV(
    bagging_svm,
    param_grid=bagging_svm_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

bag_tree_search = GridSearchCV(
    bagging_tree,
    param_grid=bagging_tree_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

bag_svm_search.fit(X_train, y_train)
bag_tree_search.fit(X_train, y_train)

best_bag_svm = bag_svm_search.best_estimator_
best_bag_tree = bag_tree_search.best_estimator_

print('Best Bagging SVM params:', bag_svm_search.best_params_)
print(f"Best CV F1 (Bagging SVM): {bag_svm_search.best_score_:.4f}")
print('Best Bagging Tree params:', bag_tree_search.best_params_)
print(f"Best CV F1 (Bagging Tree): {bag_tree_search.best_score_:.4f}")


In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('bag_svm', best_bag_svm),
        ('bag_tree', best_bag_tree)
    ],
    voting='soft',
    n_jobs=-1
)

voting_clf.fit(X_train, y_train)


In [ ]:
metrics_bag_svm = evaluate_model('Bagging SVM', best_bag_svm, X_test, y_test)
metrics_bag_tree = evaluate_model('Bagging Tree', best_bag_tree, X_test, y_test)
metrics_voting = evaluate_model('Voting (Bag SVM + Bag Tree)', voting_clf, X_test, y_test)

ensemble_df = pd.DataFrame([
    metrics_dt,
    metrics_svm,
    metrics_bag_svm,
    metrics_bag_tree,
    metrics_voting
])

ensemble_df_display = ensemble_df.drop(columns='y_pred').set_index('model').round(4)
ensemble_df_display


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, metrics) in zip(
    axes,
    [
        ('Bagging SVM', metrics_bag_svm),
        ('Bagging Tree', metrics_bag_tree),
        ('Voting Ensemble', metrics_voting)
    ]
):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        metrics['y_pred'],
        display_labels=class_names,
        ax=ax,
        cmap='Purples',
        colorbar=False
    )
    ax.set_title(f'{name} — Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
order = ensemble_df.set_index('model')['accuracy'].sort_values(ascending=False)
order.plot(kind='bar', color='teal')
plt.ylabel('Test accuracy')
plt.title('Accuracy comparison on the test set')
plt.ylim(0.9, 1.01)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
def model_size_description(name, metrics):
    if name.startswith('Decision Tree'):
        return f"Nodes: {best_dt_pipeline.named_steps['tree'].tree_.node_count}"
    if name.startswith('SVM') and 'Bagging' not in name:
        svc = best_svm.named_steps['svc']
        return f"Support vectors: {svc.support_vectors_.shape[0]}"
    if name.startswith('Bagging SVM'):
        base_svc = best_bag_svm.estimator.named_steps['svc']
        return f"Bag size {best_bag_svm.n_estimators}, support vectors ≈ {base_svc.support_vectors_.shape[0]} each"
    if name.startswith('Bagging Tree'):
        base_tree = best_bag_tree.estimator
        return f"Bag size {best_bag_tree.n_estimators}, nodes/base tree ≈ {base_tree.tree_.node_count}"
    if name.startswith('Voting'):
        return 'Combination of Bagging SVM + Bagging Tree'
    return 'N/A'

summary_table = ensemble_df.drop(columns='y_pred').copy()
summary_table['model_size'] = summary_table.apply(lambda row: model_size_description(row['model'], row), axis=1)
summary_table['interpretability'] = summary_table['model'].map({
    'Decision Tree (best)': 'High (explicit rules)',
    'SVM (best)': 'Medium (support vectors)',
    'Bagging SVM': 'Low (ensemble of SVMs)',
    'Bagging Tree': 'Medium (averaged trees)',
    'Voting (Bag SVM + Bag Tree)': 'Medium (ensemble summary)'
})

summary_display = summary_table.set_index('model').round(4)
summary_display


### Discussion
- **Performance.** Bagging SVM reaches the highest accuracy/F1, closely followed by the soft-voting hybrid, in line with the expected results.
- **Inference time.** Ensembles and SVMs are slower due to multiple base learners / support vectors, whereas the standalone tree is fastest.
- **Interpretability.** Single trees remain the most interpretable; ensembles trade transparency for accuracy.

Possible extensions include PCA visualisations or stacking ensembles as suggested in the statement.